# Timestep-Flexible S5 Reconstruction

Minimal Colab notebook for training the timestep-flexible Brain2Text24 S5 decoder and plotting `20 ms` / `40 ms` validation diagnostics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl')
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only || true

%cd {REPO_DIR}
!pip install -q torch pandas matplotlib

RAW_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1')
SMOOTHED_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1_smoothed_sigma2p0')
TRIM_SCRIPT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments' / 'trim_area6v_cache.py'
OUTPUT_ROOT = Path('/content/drive/MyDrive/utah_ssl/outputs/timestep_flexible_ssm')
EXPERIMENTS_DIR = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
PYTHONPATH = str(EXPERIMENTS_DIR)
os.environ['PYTHONPATH'] = PYTHONPATH
if PYTHONPATH not in sys.path:
    sys.path.insert(0, PYTHONPATH)

for cache_root in (RAW_CACHE_ROOT, SMOOTHED_CACHE_ROOT):
    dataset_root = cache_root / 'brain2text24'
    if dataset_root.exists():
        subprocess.run(
            ['python', str(TRIM_SCRIPT), '--cache-root', str(cache_root)],
            check=True,
            cwd=str(REPO_DIR),
        )

print('RAW_CACHE_ROOT:', RAW_CACHE_ROOT)
print('SMOOTHED_CACHE_ROOT:', SMOOTHED_CACHE_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
from pathlib import Path

RUN_NAME = 'timestep_flexible_s5_tx_only_colab'
DATASET = 'brain2text24'
FEATURE_MODE = 'tx_only'
CACHE_ROOT = RAW_CACHE_ROOT
MAX_STEPS = 12000
BATCH_SIZE = 64
LEARNING_RATE = 1e-2
MIN_LEARNING_RATE = 1e-4
ADAM_EPSILON = 1e-1
SESSION_ADAPTER = True
NORMALIZATION_MODE = 'global'
TRAIN_BIN_SIZE_MS = 20
EVAL_BIN_SIZES_MS = (20, 40)
PATCH_SIZE_MS = 280
PATCH_STRIDE_MS = 80
RESUME_LATEST = False

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUN_DIR = OUTPUT_ROOT / RUN_NAME
progress_path = RUN_DIR / 'progress.jsonl'
summary_path = RUN_DIR / 'summary.json'

assert progress_path.exists(), f'Missing: {progress_path}'

progress = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
df = pd.DataFrame(progress)

train_df = df[df.get('event').eq('timestep_flexible_train_report')] if 'event' in df.columns else pd.DataFrame()
val_df = df[df.get('event').eq('timestep_flexible_val_report')] if 'event' in df.columns else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if not train_df.empty and 'step' in train_df and 'train_ctc_bpphone' in train_df:
    axes[0].plot(train_df['step'], train_df['train_ctc_bpphone'], label='train CTC')
if not val_df.empty:
    if 'val_20ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_20ms_ctc_bpphone'], marker='o', label='val 20 ms CTC')
    if 'val_40ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_40ms_ctc_bpphone'], marker='o', label='val 40 ms CTC')
axes[0].set_title('Timestep-Flexible S5 CTC')
axes[0].set_xlabel('step')
axes[0].set_ylabel('bits / phoneme')
axes[0].legend()

if not val_df.empty:
    if 'val_20ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_20ms_phoneme_error_rate'], marker='o', label='val 20 ms PER')
    if 'val_40ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_40ms_phoneme_error_rate'], marker='o', label='val 40 ms PER')
axes[1].set_title('Timestep-Flexible S5 PER')
axes[1].set_xlabel('step')
axes[1].set_ylabel('PER')
axes[1].legend()

plt.tight_layout()
plt.show()

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary.get('metrics', {}), indent=2))


## Recompute Split Stats

Run this cell when `FEATURE_MODE` changes or after trimming/rebuilding the cache. It refreshes the canonical `20 ms` competition-train global normalization stats used by the timestep-flexible trainer. The trainer derives rebinned `40 ms` stats from the train split at runtime.

In [ ]:
import json
import subprocess

from recompute_split_feature_stats import resolve_precomputed_split_stats_path

RECOMPUTE_SPLIT_STATS = True
SPLIT_STATS_SCRIPT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments' / 'recompute_split_feature_stats.py'

stats_output_path = resolve_precomputed_split_stats_path(
    cache_root=CACHE_ROOT,
    dataset=DATASET,
    train_split_name='competition_train',
    feature_mode=FEATURE_MODE,
    preferred_path=None,
)

trim_cmd = ['python', str(TRIM_SCRIPT), '--cache-root', str(CACHE_ROOT)]
print('Running:', ' '.join(trim_cmd))
trim_result = subprocess.run(trim_cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
print('trim returncode:', trim_result.returncode)
if trim_result.stdout:
    print('\nTRIM STDOUT\n')
    print(trim_result.stdout)
if trim_result.stderr:
    print('\nTRIM STDERR\n')
    print(trim_result.stderr)
if trim_result.returncode != 0:
    raise RuntimeError('Area-6v cache trim failed before stats recompute.')

if RECOMPUTE_SPLIT_STATS:
    cmd = [
        'python', str(SPLIT_STATS_SCRIPT),
        '--cache-root', str(CACHE_ROOT),
        '--dataset', DATASET,
        '--feature-mode', FEATURE_MODE,
        '--output-path', str(stats_output_path),
        '--overwrite',
    ]
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
    print('stats returncode:', result.returncode)
    if result.stdout:
        print('\nSTATS STDOUT\n')
        print(result.stdout)
    if result.stderr:
        print('\nSTATS STDERR\n')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Split-stat recompute failed.')

print('stats_output_path:', stats_output_path)


In [ ]:
import os
import shlex

PACKAGE_ROOT = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'

cmd = [
    'python', '-m', 'timestep_flexible_ssm.train',
    '--cache-root', str(CACHE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--dataset', DATASET,
    '--feature-mode', FEATURE_MODE,
    '--normalization-mode', NORMALIZATION_MODE,
    '--train-bin-size-ms', str(TRAIN_BIN_SIZE_MS),
    '--eval-bin-sizes-ms', ','.join(str(item) for item in EVAL_BIN_SIZES_MS),
    '--patch-size-ms', str(PATCH_SIZE_MS),
    '--patch-stride-ms', str(PATCH_STRIDE_MS),
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--min-learning-rate', str(MIN_LEARNING_RATE),
    '--adam-epsilon', str(ADAM_EPSILON),
    '--val-every-steps', '100',
    '--checkpoint-every-steps', '500',
    '--progress-every-steps', '25',
]
if RESUME_LATEST:
    cmd.append('--resume-latest')
if not SESSION_ADAPTER:
    cmd.append('--disable-session-adapter')
full_cmd = ' '.join(shlex.quote(part) for part in cmd)
pythonpath = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"

print(full_cmd)
!cd {PACKAGE_ROOT} && PYTHONPATH={pythonpath} {full_cmd}


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

summary_path = RUN_DIR / 'summary.json'
progress_path = RUN_DIR / 'progress.jsonl'
summary = json.loads(summary_path.read_text())
progress = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
progress_df = pd.DataFrame(progress)
train_df = progress_df[progress_df['event'] == 'timestep_flexible_train_report'].copy()
val_df = progress_df[progress_df['event'] == 'timestep_flexible_val_report'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
if not train_df.empty:
    axes[0].plot(train_df['step'], train_df['train_ctc_bpphone'], label='train CTC')
if not val_df.empty:
    if 'val_20ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_20ms_ctc_bpphone'], marker='o', label='val 20 ms CTC')
    if 'val_40ms_ctc_bpphone' in val_df:
        axes[0].plot(val_df['step'], val_df['val_40ms_ctc_bpphone'], marker='o', label='val 40 ms CTC')
    if 'val_20ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_20ms_phoneme_error_rate'], marker='o', label='val 20 ms PER')
    if 'val_40ms_phoneme_error_rate' in val_df:
        axes[1].plot(val_df['step'], val_df['val_40ms_phoneme_error_rate'], marker='o', label='val 40 ms PER')
axes[0].set_title('Timestep-Flexible S5 CTC')
axes[0].set_xlabel('step')
axes[0].set_ylabel('bits / phoneme')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].set_title('Timestep-Flexible S5 PER')
axes[1].set_xlabel('step')
axes[1].set_ylabel('PER')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

print('best step:', summary.get('best_step'))
print('final metrics:', json.dumps(summary.get('metrics', {}), indent=2))
print('best metrics:', json.dumps(summary.get('best_metrics', {}), indent=2))
if not val_df.empty:
    columns = ['step']
    for name in [
        'val_20ms_ctc_bpphone',
        'val_40ms_ctc_bpphone',
        'val_20ms_phoneme_error_rate',
        'val_40ms_phoneme_error_rate',
    ]:
        if name in val_df.columns:
            columns.append(name)
    display(val_df[columns].tail(10))
